# Object Detection in Video using YOLO11n

**Student:** Ponnada Likhitha  
**Student No.:** 23341A1293  
**Project No.:** 32  
**Assigned Model:** YOLO11n

This notebook follows the structure specified in the attached pre-trained-model assignment. The project uses a pre-trained YOLO11n model for inference on video frames; it does not train a deep-learning model from scratch.

## 3. Problem Statement

Detect objects continuously in a video stream using the pre-trained YOLO11n object-detection model. The system reads a recorded video frame by frame, performs inference, processes the detection results, and produces an annotated output video.

## 4. Objective

- Understand a pre-trained object-detection model.
- Load YOLO11n.
- Process video frame by frame.
- Perform inference continuously.
- Display bounding boxes, class labels and confidence scores.
- Save the processed video.
- Demonstrate at least three different video test cases.

## 5. Introduction to the Pre-trained Model

YOLO (You Only Look Once) is an object-detection model family. YOLO11n is the assigned nano detection model used here for inference.

The model accepts visual frames and produces object detections. The project uses existing pre-trained weights rather than training from scratch.

Official documentation: https://docs.ultralytics.com/models/yolo11

## 6. Why This Model Was Selected

YOLO11n is the model assigned to Project 32 in the project allocation table. It is appropriate for continuous frame-by-frame object detection and produces bounding boxes, class labels and confidence scores.

## 7. Model Architecture / Working Principle

At a high level, YOLO11n uses a backbone for visual feature extraction and a detection head for multi-scale object detection. The official configuration includes convolution layers, C3k2 blocks, SPPF, C2PSA and a Detect head.

### Application pipeline

**Input Video → Video Frame → YOLO11n → Inference → Bounding Boxes / Labels / Confidence → Annotated Frame → Output Video**

## 8. Libraries and Requirements

In [ ]:
# Install once if needed
# !pip install ultralytics opencv-python numpy pandas

In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

## 9–10. Installation and Imports

The required packages are Ultralytics, OpenCV, NumPy and Pandas. The same environment can be used in VS Code or Jupyter.

## 11. Loading the Pre-trained Model

In [ ]:
MODEL_NAME = "yolo11n.pt"

model = YOLO(MODEL_NAME)

print("YOLO11n model loaded successfully.")

## 12. Input Preparation

Place three recorded videos in the `input_videos` directory:

- `test_case_1.mp4`
- `test_case_2.mp4`
- `test_case_3.mp4`

The assignment requires at least three different inputs/test cases.

In [ ]:
INPUT_DIR = Path("input_videos")
OUTPUT_DIR = Path("outputs")
RESULTS_DIR = Path("results")

OUTPUT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

videos = sorted(INPUT_DIR.glob("*.mp4"))
print("Videos found:")
for v in videos:
    print("-", v)

## 13. Pre-processing

OpenCV opens the video and supplies individual frames. Each frame is passed to the Ultralytics YOLO11n inference interface, which handles the model input preparation required for inference.

In [ ]:
def open_video(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open: {video_path}")
    return cap

## 14. Model Inference

In [ ]:
# Example inference on the first available frame
if videos:
    cap = open_video(videos[0])
    ret, frame = cap.read()
    cap.release()

    if ret:
        results = model.predict(source=frame, conf=0.25, verbose=False)
        print("Inference completed for one frame.")
    else:
        print("Could not read a frame.")

## 15. Output Processing

In [ ]:
# Draw bounding boxes, labels and confidence values
if videos and ret:
    annotated_frame = results[0].plot()
    print("Detection results were converted into an annotated frame.")

## 16. Result Visualization

In [ ]:
import matplotlib.pyplot as plt

if videos and ret:
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("YOLO11n Object Detection")
    plt.show()

## Continuous Video Detection

The following function performs inference on every frame, writes the annotated frame to an output video, and records actual detection statistics. Press `Q` when the display window is active to stop early.

In [ ]:
def process_video(video_path, output_path, show=False, conf=0.25):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 25.0

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(
        str(output_path), fourcc, fps, (width, height)
    )

    frames_processed = 0
    total_detections = 0
    classes = set()

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            results = model.predict(
                source=frame,
                conf=conf,
                verbose=False
            )

            annotated = results[0].plot()

            if results[0].boxes is not None:
                boxes = results[0].boxes
                names = results[0].names
                total_detections += len(boxes)

                for cls_id in boxes.cls.tolist():
                    classes.add(names[int(cls_id)])

            writer.write(annotated)

            if show:
                cv2.imshow("YOLO11n - Object Detection in Video", annotated)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break

            frames_processed += 1

    finally:
        cap.release()
        writer.release()
        if show:
            cv2.destroyAllWindows()

    return {
        "input_video": str(video_path),
        "output_video": str(output_path),
        "frames_processed": frames_processed,
        "total_detections": total_detections,
        "classes_detected": ", ".join(sorted(classes))
    }

## 17. Test Cases

The assignment requires at least three different inputs/test cases. Run the following cell after adding three real videos.

In [ ]:
test_cases = [
    INPUT_DIR / "test_case_1.mp4",
    INPUT_DIR / "test_case_2.mp4",
    INPUT_DIR / "test_case_3.mp4",
]

results_table = []

for path in test_cases:
    if not path.exists():
        print(f"Missing test case: {path}")
        continue

    output_path = OUTPUT_DIR / f"{path.stem}_detected.mp4"
    result = process_video(path, output_path, show=False)
    results_table.append(result)
    print(result)

results_df = pd.DataFrame(results_table)
results_df

## 18. Results / Observations

The table below must contain values produced by the actual runs. Do not invent detection counts or accuracy values.

The output videos are saved under `outputs/`.

In [ ]:
results_df.to_csv(RESULTS_DIR / "detection_summary.csv", index=False)
results_df

## 19. Limitations

- Detection quality depends on video quality.
- Small, blurred or occluded objects can be difficult to detect.
- Poor lighting can affect results.
- Processing speed depends on available hardware.
- The pre-trained model is limited to its learned object classes.

## 20. Future Scope

- Webcam-based real-time detection.
- Object tracking.
- Vehicle/person counting.
- Speed estimation.
- Domain-specific custom detection.
- GPU acceleration.
- Web application deployment.

These are future extensions; the core assignment remains object detection in video.

## 21. Conclusion

The project demonstrates practical inference using the pre-trained YOLO11n object-detection model. The video is processed frame by frame, detections are generated, and the results are converted into annotated output video with bounding boxes, class labels and confidence scores.

No deep-learning model is trained from scratch.

## Model Source and License

Official source:
- https://docs.ultralytics.com/models/yolo11
- https://github.com/ultralytics/ultralytics

YOLO11 models are provided under AGPL-3.0 and Enterprise licenses. Verify the current official licensing terms before redistribution or commercial use.

## Viva / Review Questions

1. What is a pre-trained model?
2. Why are pre-trained models used?
3. What is inference?
4. Training vs inference?
5. Pre-trained model vs training from scratch?
6. What input does YOLO11n accept?
7. What output does it produce?
8. What did the model learn during original training?
9. What is the purpose of model weights?
10. Why was YOLO11n selected?
11. What are its limitations?
12. What is the model source?
13. What license applies?
14. Does this project require adaptation?
15. What improvements could be made?